# TELCO / RETAIN: de la base a una decisión
Caso de portafolio de análisis de abandono. Una fila representa un cliente;
Churn es la etiqueta observada, no una probabilidad. Esta primera versión
no entrena modelos predictivos ni estima efectos causales.

El notebook se entrega con salidas ejecutadas mediante Python en un proceso
local. Se puede volver a ejecutar en Jupyter con pandas instalado.


In [1]:
from pathlib import Path
import sys
import sqlite3
import json
import pandas as pd

locations = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in locations if (p/'data/raw/Telco-Customer-Churn.csv').is_file()), None)
if ROOT is None:
    ROOT = next((p/'outputs/proyecto_2_retencion' for p in locations
                 if (p/'outputs/proyecto_2_retencion/data/raw/Telco-Customer-Churn.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abre el notebook desde la carpeta del proyecto.')
sys.path.insert(0, str(ROOT/'src'))
from analyze import clean
raw = pd.read_csv(ROOT/'data/raw/Telco-Customer-Churn.csv', dtype=str, keep_default_na=False)
print('Dimensiones de la fuente:', raw.shape)
print('Columnas:', ', '.join(raw.columns))


Dimensiones de la fuente: (7043, 21)
Columnas: customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, Churn


## 1. Auditar antes de transformar
Los espacios de TotalCharges no equivalen a cero. Se inspeccionan los
registros y se preservan todos los clientes. Las categorías estructurales
'No internet service' y 'No phone service' no significan lo mismo que 'No'.


In [2]:
blank_totals = raw.TotalCharges.str.strip().eq('')
print('IDs duplicados sin distinguir mayúsculas:', raw.customerID.str.upper().duplicated().sum())
print('Totales vacíos:', blank_totals.sum())
print(raw.loc[blank_totals, ['tenure', 'Churn']].value_counts().to_string())
print('\nCategorías estructurales de soporte:')
print(raw.TechSupport.value_counts().to_string())


IDs duplicados sin distinguir mayúsculas: 0
Totales vacíos: 11
tenure  Churn
0       No       11

Categorías estructurales de soporte:
TechSupport
No                     3473
Yes                    2044
No internet service    1526


## 2. Limpiar sin alterar la fuente
clean() valida el esquema y las categorías, convierte importes con Decimal,
mantiene los nulos y crea los campos de análisis. Los centavos enteros
permiten conciliar importes sin errores de representación binaria.


In [3]:
customers = clean(raw)
assert len(customers) == len(raw)
assert customers.customerID.is_unique
assert customers.TotalCharges.isna().sum() == blank_totals.sum()
print(customers[['tenure','MonthlyCharges','TotalCharges']].describe().round(2).to_string())
print('\nTramos de antigüedad:')
print(customers.TenureBand.value_counts().to_string())


        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00        7032.0
mean     32.37           64.76        2283.3
std      24.56           30.09       2266.77
min       0.00           18.25          18.8
25%       9.00           35.50        401.45
50%      29.00           70.35       1397.48
75%      55.00           89.85       3794.74
max      72.00          118.75        8684.8

Tramos de antigüedad:
TenureBand
49+ meses      2239
25-48 meses    1594
0-6 meses      1481
13-24 meses    1024
7-12 meses      705


## 3. Definir el denominador
La tasa descriptiva es clientes con Churn=Yes / todos los clientes del
contexto. No existe una fecha de corte en este archivo para reconstruir
una tasa mensual comparable entre periodos.


In [4]:
total = len(customers)
churned = int(customers.ChurnFlag.sum())
rate = churned / total
charges_churned = customers.loc[customers.ChurnFlag.eq(1), 'MonthlyChargesCents'].sum() / 100
print(f'Clientes: {total:,}; abandonos: {churned:,}; tasa: {rate:.2%}')
print(f'Cargos mensuales asociados a abandonos: {charges_churned:,.2f} UM')
print('Esta suma no es una pérdida contable ni ingreso futuro en riesgo.')


Clientes: 7,043; abandonos: 1,869; tasa: 26.54%
Cargos mensuales asociados a abandonos: 139,130.85 UM
Esta suma no es una pérdida contable ni ingreso futuro en riesgo.


## 4. Consultar una base real en SQLite
La base se abre en modo de solo lectura. Cada consulta vive en sql/ y su
resultado también se entrega como CSV. Así otra persona puede auditarlo.


In [5]:
connection = sqlite3.connect((ROOT/'database/telco_retention.sqlite').as_uri() + '?mode=ro', uri=True)
def query(filename):
    return pd.read_sql_query((ROOT/'sql'/filename).read_text(encoding='utf-8'), connection)
overview = query('01_overview.sql').iloc[0]
assert int(overview.Customers) == total
assert int(overview.Churned) == churned
assert abs(overview.MonthlyChargesChurned - charges_churned) < 0.000001
print(query('02_contract.sql').round(4).to_string(index=False))
print('\nAntigüedad:')
print(query('03_tenure.sql').round(4).to_string(index=False))


ContractLabel  Customers  Churned  ChurnRate  MonthlyChargesChurned
      Mensual       3875     1655     0.4271              120847.10
       Un ano       1473      166     0.1127               14118.45
     Dos anos       1695       48     0.0283                4165.30

Antigüedad:
 TenureBand  Customers  Churned  ChurnRate  MonthlyChargesChurned
  0-6 meses       1481      784     0.5294               49896.10
 7-12 meses        705      253     0.3589               19058.15
13-24 meses       1024      294     0.2871               23081.65
25-48 meses       1594      325     0.2039               27462.50
  49+ meses       2239      213     0.0951               19632.45


## 5. Factores asociados y comparación válida
El soporte se compara solo entre clientes con internet. Ver más abandono
sin soporte no demuestra que ofrecer soporte evitará ese abandono.
Contrato y antigüedad pueden explicar parte de la diferencia.


In [6]:
factors = query('04_factors.sql')
print(factors[factors.Factor.isin(['Internet','Soporte (solo internet)','Metodo de pago'])].round(4).to_string(index=False))
print('\nSoporte dentro de cada contrato (también descriptivo):')
print(query('10_support_within_contract.sql').round(4).to_string(index=False))


                 Factor                 Category  Customers  Churned  ChurnRate  MonthlyChargesChurned
               Internet             Fibra optica       3096     1297     0.4189              114300.05
               Internet                      DSL       2421      459     0.1896               22529.20
               Internet             Sin internet       1526      113     0.0740                2301.60
         Metodo de pago       Cheque electronico       2365     1071     0.4529               84288.75
         Metodo de pago        Cheque por correo       1612      308     0.1911               16803.60
         Metodo de pago Transferencia automatica       1544      258     0.1671               20091.90
         Metodo de pago       Tarjeta automatica       1522      232     0.1524               17946.60
Soporte (solo internet)              Sin soporte       3473     1446     0.4164              110709.80
Soporte (solo internet)              Con soporte       2044      310     

## 6. Cruces, importe y tamaño de muestra
Los cruces pequeños no deben liderar un ranking por una tasa extrema.
Los tramos de cargos (<35, 35 a <80 y >=80 UM) son cortes exploratorios
documentados, no umbrales validados del negocio.


In [7]:
print(query('05_contract_tenure.sql').round(4).to_string(index=False))
print('\nDistribución por tramo de cargo y abandono:')
print(query('08_charges.sql').round(2).to_string(index=False))


ContractLabel  TenureBand  Customers  Churned  ChurnRate                         SampleNote
      Mensual   0-6 meses       1413      780     0.5520                        Base >= 100
      Mensual  7-12 meses        581      244     0.4200                        Base >= 100
      Mensual 13-24 meses        737      278     0.3772                        Base >= 100
      Mensual 25-48 meses        802      264     0.3292                        Base >= 100
      Mensual   49+ meses        342       89     0.2602                        Base >= 100
       Un ano   0-6 meses         39        4     0.1026 Base pequena: lectura exploratoria
       Un ano  7-12 meses         85        9     0.1059 Base pequena: lectura exploratoria
       Un ano 13-24 meses        197       16     0.0812                        Base >= 100
       Un ano 25-48 meses        518       55     0.1062                        Base >= 100
       Un ano   49+ meses        634       82     0.1293                        

## 7. Seis segmentos excluyentes
En contratos mensuales se aplica una jerarquía: primero <=6 meses;
luego fibra sin soporte; luego cargo >=80 UM; finalmente el resto.
Los contratos anual y bianual forman otros dos grupos. Un cliente
pertenece a un solo segmento. Los datos demográficos no deciden elegibilidad.


In [8]:
segments = pd.read_csv(ROOT/'analysis/priority_segments.csv')
assert int(segments.Customers.sum()) == total
assert int(segments.Churned.sum()) == churned
print(segments[['SegmentName','Customers','Churned','ChurnRate','Active','PriorityLabel']].round(4).to_string(index=False))
print('\nReglas:')
print(segments[['SegmentID','Rule']].to_string(index=False))


            SegmentName  Customers  Churned  ChurnRate  Active PriorityLabel
01 | Bienvenida mensual       1413      780     0.5520     633      Piloto 1
 02 | Fibra sin soporte       1227      607     0.4947     620      Piloto 2
03 | Revision de tarifa        281       92     0.3274     189      Piloto 3
  04 | Mensual restante        954      176     0.1845     778   Seguimiento
    05 | Contrato anual       1473      166     0.1127    1307   Seguimiento
  06 | Contrato bianual       1695       48     0.0283    1647   Seguimiento

Reglas:
SegmentID                                          Rule
       S1          Mensual y antiguedad de 0 a 6 meses.
       S2 Mensual, mas de 6 meses, fibra y sin soporte.
       S3  Mensual restante con cargo mensual >= 80 UM.
       S4                 Resto de contratos mensuales.
       S5                           Contrato de un ano.
       S6                         Contrato de dos anos.


## 8. Candidatos activos, no probabilidades de abandono
La elegibilidad de piloto exige base >=100, activos >=50 y tasa observada
superior a la global. El orden usa abandonos observados. Es una heurística
exploratoria que debe validarse; no es un modelo predictivo.


In [9]:
candidates = query('07_campaign_candidates.sql')
assert candidates.customerID.is_unique
assert set(candidates.customerID).issubset(set(customers.loc[customers.ChurnFlag.eq(0),'customerID']))
print('Clientes activos candidatos:', len(candidates))
print(candidates.groupby('SegmentName').size().to_string())
print('\nLos IDs son del dataset de muestra. No hay datos de contacto ni campaña enviada.')


Clientes activos candidatos: 1442
SegmentName
01 | Bienvenida mensual    633
02 | Fibra sin soporte     620
03 | Revision de tarifa    189

Los IDs son del dataset de muestra. No hay datos de contacto ni campaña enviada.


## 9. Sensibilidad: supuestos separados de resultados
Contactos x mejora incremental supuesta x cargo mensual medio del segmento.
La diferencia resta solo coste de contacto: no es beneficio neto, pues faltan
margen, coste de servicio e incentivos. El tamaño 200 es ilustrativo,
no un cálculo de potencia estadística.


In [10]:
scenarios = pd.read_csv(ROOT/'analysis/scenarios.csv')
print(scenarios[scenarios.SegmentID.eq('S1')].round(2).to_string(index=False))
connection.close()


SegmentID             SegmentName  Targets  AssumedIncrementalRetention  AvgMonthlyChargeActive  IllustrativeGrossMonthlyCharges  IllustrativeContactCost  GrossLessContactCostOnly                                               Label
       S1 01 | Bienvenida mensual      200                         0.01                   46.25                            92.49                    200.0                   -107.51 Simulacion; no beneficio neto ni impacto demostrado
       S1 01 | Bienvenida mensual      200                         0.03                   46.25                           277.47                    200.0                     77.47 Simulacion; no beneficio neto ni impacto demostrado
       S1 01 | Bienvenida mensual      200                         0.05                   46.25                           462.45                    200.0                    262.45 Simulacion; no beneficio neto ni impacto demostrado


## 10. Recomendación y siguiente validación
Empezar por un piloto de bienvenida en clientes mensuales nuevos, con
asignación aleatoria y control, previa verificación de consentimiento y
capacidad. Recoger fechas y resultados a 30/60/90 días. Separar efectos
medidos de asociaciones de esta muestra. Antes de entrenar un modelo,
definir horizonte, disponibilidad temporal de variables y validación.

Fuente de referencia: [IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv).
La copia utilizada es la aportada por el usuario, identificada por SHA-256
en analysis/source_manifest.json; no se comprobó identidad con el remoto.
